<a href="https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### 1. My Lane as an ML Task (Type)

**Task Type:** Ranking / Scoring (Supported by Supervised Binary Classification)

**Why?** An editorial team has limited human review bandwidth and cannot inspect thousands of pages simultaneously. The business goal is not merely to classify pages into binary "good" or "bad" buckets, but to output a **prioritized, ranked queue** where pages with high decay risk and high business exposure appear at the top. The underlying model estimates the probability of traffic decay, which is combined with exposure metrics to produce an actionable priority score.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load dataset via direct GitHub raw URL
raw_url = "https://raw.githubusercontent.com/hamzakhattak-AIDev/flyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(raw_url)

print(f"Task Definition: Ranking {len(df):,} pages into an actionable priority queue.")

Task Definition: Ranking 30,000 pages into an actionable priority queue.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### 2. Target Definition & Nature

* **Target Variable:** `is_declining_label` ($1$ if `trend_direction == "down"`, $0$ otherwise).
* **Target Type:** **Observed Proxy Outcome**.
* **Source & Validity:** The label reflects an observed trend calculated from historical traffic changes rather than a human-defined opinion or synthetic rule.
* **Capstone Upgrade Path:** In the starter dataset, this label acts as a proxy for decay within the current window. For our full capstone pipeline on the warehouse, we upgrade this target to an explicit, forward-looking outcome: using a 90-day feature window to predict observed traffic decline over the subsequent 30-day window.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create and audit the target label distribution
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

target_counts = df["is_declining_label"].value_counts()
target_pct = df["is_declining_label"].mean()

print(f"Target Label Breakdown:")
print(f"Stable / Growing (0): {target_counts[0]:,} pages ({1-target_pct:.1%})")
print(f"Declining (1):        {target_counts[1]:,} pages ({target_pct:.1%})")

Target Label Breakdown:
Stable / Growing (0): 13,738 pages (45.8%)
Declining (1):        16,262 pages (54.2%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### 3. Primary Success Metric

* **Primary Metric:** **Precision@K** (specifically **Precision@20** and **Precision@50**).
* **Why this metric?** Generic accuracy is uninformative due to class imbalance, and ROC-AUC evaluates the entire list (including low-ranking tail pages that editors will never see). Since editorial teams can only review $K$ pages per week, Precision@K measures the exact fraction of true declining/decaying pages present within the top $K$ flagged recommendations.
* **Secondary Metric:** Average Precision (PR-AUC) to assess overall ranking quality.
* **What means 'good'?** Beating the baseline hand rule ($\text{Precision}@50 \approx 0.240$) by achieving a $\text{Precision}@50 \ge 0.600$ (meaning at least 30 out of the top 50 flagged pages are true decay candidates).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Random baseline check
y = df["is_declining_label"].values
random_scores = np.random.rand(len(y))

print(f"Baseline Random Precision@20: {precision_at_k(random_scores, y, 20):.3f}")
print(f"Baseline Random Precision@50: {precision_at_k(random_scores, y, 50):.3f}")
print(f"Target Benchmark Precision@50: >= 0.600")

Baseline Random Precision@20: 0.500
Baseline Random Precision@50: 0.580
Target Benchmark Precision@50: >= 0.600


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### 4. Unit of Analysis

**Unit of Analysis:** One row = **One Pseudonymized Content Item** (`content_id` / `content_hash_id`) evaluated for a single client over a 90-day performance window.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Displaying the exact unit of analysis dataframe structure
sample_unit = df[["content_id", "client_id", "content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "is_declining_label"]].head(5)

print(f"Grain: 1 Row = 1 Content Unit ({df['content_id'].nunique():,} unique units)")
sample_unit

Grain: 1 Row = 1 Content Unit (30,000 unique units)


,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,is_declining_label
0,content_304f48230142,client_f369cb89fc,187,20,3803,10.6,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,445,25,15320,20.3,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,141,20,12581,36.5,0.09,1
3,content_331d6c4de07b,client_19581e27de,463,22,11751,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,263,14,19140,44.0,0.13,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### 5. Why ML Beats a Fixed Rule

A fixed `if/else` rule (such as `days_since_last_update >= 180 AND impressions >= 500`) captures obvious low-hanging fruit, but it breaks down under messy real-world search conditions:

1. **Non-Linear Interactions:** Page age alone does not cause traffic decay. A 2-year-old high-intent evergreen guide might remain stable, whereas a 90-day-old news piece might decay rapidly. ML captures subtle non-linear interactions between content age, CTR deviations, and position tiers.
2. **SERP Layout & Intent Variance:** Click-through rates vary drastically across SERP position tiers (e.g., Position 2 vs Position 8). A simple rule cannot easily adjust thresholds dynamically for every ranking tier without becoming an unmaintainable wall of hand-tuned code.
3. **Smooth Prioritization:** Simple heuristics output binary flags or crude tie-filled groups. Machine learning outputs continuous probability scores, allowing us to generate a smooth, granular rank-order list that maximizes Precision@K for any team capacity.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Demonstrating rule limitations: Stale pages aren't always declining!
stale_df = df[df["days_since_last_update"] >= 180]
stale_decline_rate = stale_df["is_declining_label"].mean()

print(f"Total Stale Pages (>=180d): {len(stale_df):,}")
print(f"Fraction of Stale Pages actually declining: {stale_decline_rate:.1%}")
print("Conclusion: Simple 'age rules' miss over 50% of cases and create false positives. ML is needed to untangle signal from noise.")

Total Stale Pages (>=180d): 174
Fraction of Stale Pages actually declining: 47.1%
Conclusion: Simple 'age rules' miss over 50% of cases and create false positives. ML is needed to untangle signal from noise.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.